In [ ]:
import lmstudio as lms

model = lms.llm("qwen/qwen3-vl-4b")

In [ ]:
import lmstudio as lms

image_path = "feature_importance.png" # Replace with the path to your image
image_handle = lms.prepare_image(image_path)

In [ ]:
import lmstudio as lms

chat = lms.Chat()
chat.add_user_message("Describe this image please", images=[image_handle])
prediction = model.respond(chat)

In [ ]:
prediction

In [17]:
from IPython.display import display, Markdown as IPyMarkdown
import re

# Çıktıyı al
content = prediction.content

# ============================================
# SADECE LaTeX FORMATLI MARKDOWN ÇIKTISI
# ============================================

# Başlık çıkarma - dinamik
title_match = re.search(r'["\'](.+?)["\']', content.split('\n')[0])
title = title_match.group(1) if title_match else "Vision Model Analysis"

display(IPyMarkdown(f"# 🎯 {title}"))
display(IPyMarkdown("---"))

# Metin paragraflarını ayır
paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]

# Overview
display(IPyMarkdown("## 📊 Overview"))
for i, para in enumerate(paragraphs[:2]):
    if not para.startswith('-') and not para.startswith('*'):
        display(IPyMarkdown(para))

# Liste öğelerini ve sayısal verileri ayıkla
list_items = []
for para in paragraphs:
    items = re.findall(r'^[\s]*[-*•]\s*(.+?)$', para, re.MULTILINE)
    list_items.extend(items)

# Sayısal değerleri çıkar
data_pattern = r'\*\*([^*]+?)\*\*.+?(?:approximately|around|about|score of)\s+(\d+\.?\d*)'
data_matches = []

for item in list_items:
    match = re.search(data_pattern, item, re.IGNORECASE)
    if match:
        feature_name = match.group(1).strip()
        value = float(match.group(2))
        data_matches.append((feature_name, value))

# Eğer veri bulunamazsa alternatif pattern dene
if not data_matches:
    alt_pattern = r'`?([^`\n]+?)`?\s*\(approx\.\s*(\d+\.?\d+)\)'
    for item in list_items:
        match = re.search(alt_pattern, item)
        if match:
            feature_name = match.group(1).strip()
            value = float(match.group(2))
            data_matches.append((feature_name, value))

# Veriyi sırala
data_matches.sort(key=lambda x: x[1], reverse=True)

# Tablo oluştur (eğer veri varsa)
if len(data_matches) >= 3:
    display(IPyMarkdown("## 📊 Feature Importance Data"))
    
    table_md = "| Rank | Feature Name | Importance Score | LaTeX |\n"
    table_md += "|:----:|:-------------|:----------------:|:-----:|\n"
    
    for idx, (name, value) in enumerate(data_matches[:15], 1):
        latex_val = f"$\\approx {value:.2f}$"
        table_md += f"| {idx} | `{name}` | {value:.2f} | {latex_val} |\n"
    
    display(IPyMarkdown(table_md))

# Matematiksel analiz (eğer sayısal veriler varsa)
if data_matches:
    display(IPyMarkdown("## 📐 Mathematical Analysis"))
    
    values = [v for _, v in data_matches]
    total = sum(values)
    max_val = max(values)
    min_val = min(values)
    mean_val = total / len(values)
    variance = sum((x - mean_val) ** 2 for x in values) / len(values)
    std_dev = variance ** 0.5
    
    # İstatistiksel analiz
    stats_md = f"""
### 📈 Statistical Summary

| Metric | Formula | Value |
|:-------|:--------|------:|
| **Sample Size** | $N$ | ${len(values)}$ |
| **Total Sum** | $\\sum x_i$ | ${total:.4f}$ |
| **Range** | $[x_{{\\min}}, x_{{\\max}}]$ | $[{min_val:.4f}, {max_val:.4f}]$ |
| **Mean** | $\\mu = \\frac{{\\sum x_i}}{{N}}$ | ${mean_val:.4f}$ |
| **Variance** | $\\sigma^2 = \\frac{{\\sum(x_i - \\mu)^2}}{{N}}$ | ${variance:.4f}$ |
| **Std Deviation** | $\\sigma = \\sqrt{{\\sigma^2}}$ | ${std_dev:.4f}$ |
| **Coefficient of Variation** | $CV = \\frac{{\\sigma}}{{\\mu}}$ | ${std_dev/mean_val:.4f}$ |

### 🏆 Top 3 Features Analysis

The top 3 features combined contribution:

$$
\\text{{Top 3 Sum}} = {values[0]:.4f} + {values[1]:.4f} + {values[2]:.4f} = {sum(values[:3]):.4f}
$$

Percentage of total importance:

$$
\\text{{Percentage}} = \\frac{{{sum(values[:3]):.4f}}}{{{total:.4f}}} \\times 100\\% = {(sum(values[:3])/total)*100:.2f}\\%
$$

Dominance ratio (1st vs 2nd feature):

$$
\\text{{Dominance}} = \\frac{{{values[0]:.4f}}}{{{values[1]:.4f}}} = {values[0]/values[1]:.2f}\\times
$$

> **Interpretation**: The top feature is **{values[0]/values[1]:.2f}x** more important than the second feature.
"""
    
    display(IPyMarkdown(stats_md))

# Key insights
display(IPyMarkdown("## 🔍 Key Insights"))

insights = []
for para in paragraphs:
    if any(keyword in para.lower() for keyword in ['summary', 'influenced', 'predictor', 'significant']):
        # Paragrafı temizle
        clean_para = para.replace('**', '').strip('- *')
        insights.append(clean_para)

if insights:
    for i, insight in enumerate(insights[:3], 1):
        display(IPyMarkdown(f"**{i}.** {insight}"))
else:
    # Varsayılan insight
    display(IPyMarkdown("Analysis shows varying levels of feature importance in the model."))

# Son not
display(IPyMarkdown("---"))
display(IPyMarkdown(f"*Analysis generated from {len(paragraphs)} paragraphs with {len(data_matches)} numerical data points.*"))

# 🎯 Feature Importance from Random Forest

---

## 📊 Overview

This is a bar chart titled "Feature Importance from Random Forest". It displays the relative importance of different features in a Random Forest model, likely for predicting something like housing prices or property values.

The x-axis lists the feature names, and the y-axis represents the importance score, ranging from 0.0 to 0.6.

...

## 🔍 Key Insights

**1.** income_per_capita is the most important feature, with a score of approximately 0.60. This suggests that per capita income is the strongest predictor in the model.
- latitude is the second most important feature, with a score of approximately 0.10.
- longitude is the third most important, also around 0.10.
- The remaining features have much lower importance scores, all below 0.05. These include:
  - `distance_to_ocean` (approx. 0.04)
  - `housing_median_age` (approx. 0.04)
  - `median_income` (approx. 0.03)
  - `population_per_household` (approx. 0.03)
  - `rooms_per_household` (approx. 0.02)
  - `bedrooms_ratio` (approx. 0.02)
  - `population` (approx. 0.01)
  - `total_rooms` (approx. 0.01)
  - `total_bedrooms` (approx. 0.01)
  - `households` (approx. 0.01)

**2.** In summary, the model's predictions are heavily influenced by per capita income, followed by geographic location (latitude and longitude). Other factors like distance to the ocean, housing age, and population metrics are much less significant.

---

*Analysis generated from 5 paragraphs with 3 numerical data points.*